# 04 — Aprimoramento da seta de +15 minutos

Esta etapa testa 30 épocas, paciência, diferenças de CGM, histórico relativo e perda ponderada, incluindo combinações. O algoritmo v1 e seus resultados ficam preservados. Este notebook reúne **artefatos de execuções reais**; executar suas células refaz a leitura dos resultados, sem iniciar novamente a bateria de treinamentos.

A triagem compara todas as famílias em B3/semente 42. A confirmação compara referência e finalista nos blocos/sementes registrados. Escalas são ajustadas no treino; `stop` escolhe a época; `evaluation` compara configurações. Somente os primeiros 80% de cada série entram no desenvolvimento. O teste consultado anteriormente continua exploratório.

Implementação, execução e análise assistidas por IA a pedido da equipe. Consulte `docs/analises/ANALISE_APRIMORAMENTO_V2.md` para a interpretação e as limitações.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').exists())
EXPERIMENT = ROOT / 'results/azt1d_seta/experiments/aprimoramento_v2'
protocol = json.loads((EXPERIMENT / 'protocolo_execucao.json').read_text(encoding='utf-8'))
display(pd.DataFrame(protocol['variants']).T)
display(pd.read_csv(EXPERIMENT / 'fronteiras.csv').head(12))

## Comparações na validação temporal
As médias dão peso igual por pessoa. Triagem é provisória; o relatório de confirmação informa quais critérios passaram e quais falharam.

In [ ]:
for phase in ['triagem', 'confirmacao']:
    folder = EXPERIMENT / 'analise' / phase
    if not (folder / 'comparison.csv').exists():
        print(phase, ': ainda sem relatório completo')
        continue
    display(Markdown('### ' + phase))
    table = pd.read_csv(folder / 'comparison.csv')
    display(table[table.level.eq('overall')])
    print((folder / 'acceptance.json').read_text(encoding='utf-8'))
    display(Image(filename=str(folder / 'comparacao_aprimoramento.png')))

In [ ]:
FINAL = ROOT / 'results/azt1d_seta/runs/full_v2'
if (FINAL / 'selecao_final.json').exists():
    print((FINAL / 'selecao_final.json').read_text(encoding='utf-8'))
if (FINAL / 'final/metricas.csv').exists():
    metrics = pd.read_csv(FINAL / 'final/metricas.csv')
    display(metrics[metrics.scope.eq('macro')])
else:
    print('Sem avaliação final registrada nesta etapa.')

## Reprodução

Os comandos e fontes exatos ficam em `results/azt1d_seta/experiments/aprimoramento_v2/`. O executor `scripts/executar_aprimoramento.py` recebe variantes, blocos e sementes explicitamente, e recusa substituir uma etapa registrada. `scripts/resumir_aprimoramento.py` recalcula métricas a partir das previsões e verifica a cobertura. `scripts/validar_aprimoramento.py` verifica os artefatos e o isolamento temporal.

Uma candidata só substitui a referência se passar pelos critérios previamente registrados. Melhorar o erro geral não comprova melhora em estabilidade e extremos.